# Imports

In [24]:
import Neural_Network_module
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sklearn.metrics import mean_squared_error

# Read DF

In [9]:
filepath = r'C:\Users\mathi\OneDrive\Documents\Studia\SGH\Semestr 4\Praca dyplomowa'
df = pd.read_excel(f'{filepath}/Output/Preprocessed.xlsx',sheet_name='Preprocessed DF')
X = torch.tensor(df.values, dtype=torch.float32)
df.head()

,Operational setting 1,Operational setting 2,Ps30,P15,phi,Nc
0,-0.080460,-0.666667,-0.261905,1,0.266525,-0.780490
1,0.218391,-0.500000,-0.238095,1,0.530917,-0.799515
2,-0.494253,0.500000,-0.500000,1,0.590618,-0.719914
3,0.080460,0.000000,-0.666667,1,0.778252,-0.750965
4,-0.218391,-0.333333,-0.488095,1,0.492537,-0.700081


# Quantum Model

In [14]:
n_qubits  = len(df.iloc[0,:]) # Number of features
n_latent  = 2 # Number of latent qubits i.e. comression to 2 qubits
n_trash   = n_qubits - n_latent
n_layers  = 2          # głębokość ansatzu (warstw entanglingowych)

latent_wires = list(range(n_latent))                  # np. [0,1]
trash_wires  = list(range(n_latent, n_qubits))        # np. [2,3,4,5]
BATCH_SIZE = 870


dev = qml.device("default.qubit", wires=n_qubits, shots=None)


# Przydatne: kształt wag StronglyEntanglingLayers
weight_shape = (n_layers, n_qubits)

### Encoder

In [15]:
def encoder(weights):
    """Encoder function.
    Entangles qubits and sets their Y rotation

    Args:
        weights (_type_): _description_
    """
    # iterate over layers
    for layer in range(n_layers):
        qml.BasicEntanglerLayers(weights=weights, wires=range(n_qubits))
        # Encode rotations
        for i in range(n_qubits):
            qml.RY(weights[layer, i], wires=i)

        # Entangling CNOTs in a chain
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

### Decoder

In [16]:
def decoder(weights):
    """Decoder Function
    Entangles qubits and sets their Y rotation

    Args:
        weights (_type_): _description_
    """
   # iterate over layers
    for layer in range(n_layers):
        # Encode rotations
        for i in range(n_qubits):
            qml.RY(weights[layer, i], wires=i)

        # Entangling CNOTs in a chain
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

### Qnode Full Autoencoder

In [17]:
# @qml.qnode(dev, interface="torch", diff_method="best")
# def decode_qnode(x, encoder_weight, decoder_weight):
#     """Autoencoder QNODE

#     Args:
#         x (list): list of entry data
#         encoder_weight (torch.tensor): list of weights for the encoder
#         decoder_weight (torch.tensor): list of weights for the decoder

#     Returns:
#         torch.tensor: list of expected values of Pauli Z measurements of each cubits. Values [-1,1]
#     """
#     # Loading data into quantum state
#     qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')

#     # Encode data
#     encoder(encoder_weight)

#     # Decode data
#     decoder(decoder_weight)

#     # Perform PauliZ measurment and get expected values for tyhose measurments
#     pauli_z_exp = [qml.expval(qml.PauliZ(qubit_no))
#                    for qubit_no in range(n_qubits)]

#     return pauli_z_exp

# Batched version of decode_qnode
@qml.qnode(dev, interface="torch", diff_method="best")  # adjoint zwykle przyspiesza
def decode_qnode_batch(X_batch, encoder_weight, decoder_weight):
    qml.AngleEmbedding(
        X_batch, wires=range(n_qubits), rotation="Y")
    encoder(encoder_weight)
    decoder(decoder_weight)
    # broadcast: dostaniesz tensory o kształcie (B, n_qubits)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


### Qnode for trash wires probabilty measurement




In [18]:
# @qml.batch_params
# @qml.qnode(dev, interface="torch", diff_method="best")
# def trash_zero_qnode(x, encoder_weights):
#     """QNODE used to measure the probability of trash qubits in bottleneck layer

#     Args:
#         x (list): input data
#         encoder_weights (torch.tensor): encoder weights

#     Returns:
#         torch.tensor: list of probabilities of measuring the state |0...0> on the trash wires
#     """

#     # Load data into quantum state
#     qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')
#     encoder(encoder_weights)

#     probs = qml.probs(wires=trash_wires)
#     return probs

@qml.qnode(dev, interface="torch", diff_method="best")
def trash_zero_qnode_batch(X_batch, encoder_weight):
    qml.AngleEmbedding(
        X_batch, wires=range(n_qubits), rotation="Y")
   
    encoder(encoder_weight)
    # broadcast: kształt (B, 2**n_trash)
    probs = qml.probs(wires=trash_wires)
    return probs

### Qnode for latent state measurement




In [19]:
@qml.qnode(dev, interface="torch", diff_method="best")
def latent_state_values(x, encoder_weights):
    """Qnode used to measure the latent (nottleneck) layer

    Args:
        x (list): list of values used as entry
        encoder_weights (torch.tensor): list of weights for the encoder

    Returns:
        torch.tensor: list of expected values of Pauli Z measurements of each latent cubits. Values [-1,1]
    """
    # Load data into quantum state
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation='Y')

    # Encode data: entangler and rotations
    encoder(encoder_weights)

    # Perform PauliZ measurment and get expected values for tyhose measurments
    pauli_z_exp = [qml.expval(qml.PauliZ(w)) for w in latent_wires]

    return pauli_z_exp

### Pytorch Model

In [20]:
class QuantumAutoencoder(nn.Module):
    def __init__(self):
        """initialize en/decoder weights
        """
        super().__init__()
        # Parametry enkodera i dekodera (trainable)
        self.encoder_weight = nn.Parameter(
            torch.rand(size=weight_shape)*0.2-0.1)

        self.decoder_weight = nn.Parameter(
            torch.rand(size=weight_shape)*0.2-0.1)

    @staticmethod
    def _scale_input(x):
        """Scaling function left for legacy purposes. Returns unaltered value. Earlier it clamped on [-pi,pi]

        Args:
            x (_type_): _description_

        Returns:
            _type_: _description_
        """
        scaled = torch.clamp(x, -3.14159, 3.14159)  # legacy
        return x

    def forward(self, x):
        """Forward pass of class

        Args:
            x (torch.tensor): single row of torch tensor containing data

        Returns:
            torch.tensor, torch.tensor: list of reconstructed data, probability of trash wires being in |0...0>
        """
        single_input_flag = False
        if x.dim() == 1:
            x = x.unsqueeze(0)  # add batch dimension if missing
            single_input_flag = True

        # x_prim = decode_qnode(x, self.encoder_weight, self.decoder_weight)
        x_prim = decode_qnode_batch(x, self.encoder_weight, self.decoder_weight)

        if isinstance(x_prim, (list, tuple)):
            x_prim = torch.stack(x_prim, dim=1)

        # Measure probabilities of trash wires being in |0...0>
        probs_trash = trash_zero_qnode_batch(x, self.encoder_weight)
        # index 0 corresponds to |0...0>
        p_zero_trash = probs_trash[:, 0]

        if single_input_flag:
            x_prim = x_prim.squeeze(0)
            p_zero_trash = p_zero_trash.squeeze(0)

        return x_prim, p_zero_trash

    @staticmethod
    def loss_fn(batch, x_prim, p_zero_trash, alpha=1.0, beta=1):
        """Loss function

        Args:
            batch (torch.tensor): tensor shape [B, n_qubits]
            x_prim (torch.tensor): reconstructed tensor [B, n_qubits]
            p_zero (torch.tensor): tensor [B] of probability of trash wires being in |0...0>
            alpha (float, optional): reconstruction loss factor. Defaults to 1.0.
            beta (float, optional): compression loss factor. Defaults to 0.5.

        Returns:
            torch.tensor : 
            total loss, 
        """       
        recon_loss = torch.mean((x_prim - batch)**2)
        compression_loss = 1.0 - torch.mean(p_zero_trash)
        total_loss = alpha*recon_loss + beta*compression_loss
        return total_loss, recon_loss.item(), compression_loss.item()

        

# Read weights from file reconstruct and calculate metrics

In [25]:
model1 = Neural_Network_module.Autoencoder_Linear1()
model2 = Neural_Network_module.Autoencoder_Linear2()
model3 = Neural_Network_module.Autoencoder_Linear3()
model4 = QuantumAutoencoder()


models_dict = {
    'Autoencoder 1':
               {'Model': model1,
                'Losses':[],
                'Training time':'',
                'Epochs':''},
    'Autoencoder 2':
               {'Model': model2,
                'Losses':[],
                'Training time':'',
                'Epochs':''},
    'Autoencoder 3':
               {'Model': model3,
                'Losses':[],
                'Training time':'',
                'Epochs':''},
               }

for name in models_dict.keys():
    model = models_dict[name]['Model']
    path =filepath +f'\\Code\\Models\\{name}.pth'
    checkpoint = torch.load(path)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    
    with torch.no_grad():
        x_recon = model(X)
    df_recon = pd.DataFrame(x_recon.numpy(), columns=df.columns)

    models_dict[name]['Losses'] = checkpoint["loss"]
    models_dict[name]['Training time'] = checkpoint["training_time"]
    models_dict[name]['Epochs'] = len(checkpoint["loss"])
    models_dict[name]['Reconstructed DF'] = df_recon
    models_dict[name]['Delta DF'] = df - df_recon
    models_dict[name]['MSE'] = mean_squared_error(df.values, df_recon.values)

RECON_BATCH_SIZE = 100
reconstructions = []
model4.eval()
path =filepath +f'\\Code\\Models\\quantum_autoencoder_weights.pth'
checkpoint = torch.load(path)
model4.load_state_dict(checkpoint["model_state_dict"])

with torch.no_grad():
    for batch_start in tqdm(range(0, X.shape[0], RECON_BATCH_SIZE)):
        batch_end = min(batch_start + RECON_BATCH_SIZE, X.shape[0])
        x_batch = X[batch_start:batch_end]
        x_prim, _ = model4(x_batch)
        x_numpy = x_prim.detach().cpu().numpy()
        reconstructions.append(x_numpy)
reconstructions = np.vstack(reconstructions)
df_recon_q = pd.DataFrame(reconstructions, columns=df.columns)

# Add quantum model to dictionary
models_dict['Quantum'] = {
    'Model': model4,
    'Losses': checkpoint["loss"],
    'Training time': checkpoint["training_time"],
    'Epochs': len(checkpoint["loss"]),
    'Reconstructed DF': df_recon_q,
    'Delta DF': df - df_recon_q,
    'MSE': mean_squared_error(df.values, df_recon_q.values)
}

100%|██████████| 207/207 [00:17<00:00, 11.95it/s]


# Summary comparison

In [ ]:
def calculate_metrics(df_feature, df_recon_feature, delta_feature):
    mean = delta_feature.mean()
    median = delta_feature.median()
    max_delta = delta_feature.max()
    min_delta = delta_feature.min()
    q3, q1 = np.percentile(delta_feature, [75, 25])
    iqr = q3 - q1
    n_outliers = ((delta_feature < (q1 - 1.5 * iqr)) |
                  (delta_feature > (q3 + 1.5 * iqr))).sum()
    outlier_pct = n_outliers / len(delta_feature) * 100
    rmse = np.sqrt(mean_squared_error(df_feature, df_recon_feature))

    return {
        'RMSE': rmse,
        "Median": median,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Delta min": min_delta,
        "Delta max": max_delta,
        "Outlier %": outlier_pct
    }

In [ ]:
metrics = ["RMSE", "Median", "Q1", "Q3", "IQR", "Delta min", "Delta max", "Outlier %"]

index = pd.MultiIndex.from_product(
    [models_dict.keys(), metrics],
    names=["Model", "Metric"]
)

result_df = pd.DataFrame(
    index=index,
    columns=df.columns,
    dtype=float
)

for model_name in models_dict.keys():
    df_recon_ = models_dict[model_name]['Reconstructed DF']
    delta_df = models_dict[model_name]['Delta DF']
    for feature in delta_df.columns:
        metrics_dict = calculate_metrics(df[feature],df_recon_[feature],delta_df[feature])

        for metric, value in metrics_dict.items():
            result_df.loc[(model_name, metric), feature] = value


print(result_df)

                         Operational setting 1  Operational setting 2  \
Model         Metric                                                    
Autoencoder 1 RMSE                    0.248591               0.290190   
              median                  0.002203              -0.035025   
              Q1                     -0.165898              -0.228642   
              Q3                      0.169260               0.162115   
              IQR                     0.335158               0.390756   
              min                    -0.992218              -0.846359   
              max                     0.986104               1.173201   
              outlier_%               0.625273               0.770685   
Autoencoder 2 RMSE                    0.247917               0.460044   
              median                  0.013544               0.033275   
              Q1                     -0.148518              -0.357998   
              Q3                      0.181877     